# Notebook 4 . Bersihkan Isi CSV

Rapikan spasi, samakan penanda nilai kosong, buang baris kosong, dan (opsional) ubah kolom tanggal atau angka. File sumber tetap aman; hasil ditulis sebagai file baru.

In [ ]:
from pathlib import Path
import pandas as pd

# PENGATURAN YANG BISA DIUBAH
Input_CSV = Path("Input/input.csv")
Separator_CSV = ","  # Ganti menjadi ";" jika CSV memakai titik koma
Penanda_Nilai_Kosong = ["", "NA", "N/A", "null", "NULL", "-"]
Rapikan_Spasi = True
Kolom_Huruf_Kecil = []  # Contoh: ["email"] untuk mengubah isinya menjadi huruf kecil
Kolom_Tanggal = []       # Contoh: ["tanggal"]
Kolom_Angka = []     # Contoh: ["harga", "jumlah"]
Hapus_Baris_Kosong = True
Output_CSV = Path("Output/hasil_clean.csv")

def find_project_dir():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder / "Input").is_dir():
            (folder / "Output").mkdir(parents=True, exist_ok=True)
            return folder
    return current

BASE_DIR = find_project_dir()


## Terapkan pembersihan

In [ ]:
source = Input_CSV if Input_CSV.is_absolute() else BASE_DIR / Input_CSV
if not source.exists(): raise FileNotFoundError(f"File tidak ditemukan: {source}")
df = pd.read_csv(source, sep=Separator_CSV, dtype="string", encoding="utf-8-sig", keep_default_na=False)
df.columns = df.columns.astype(str).str.strip()
if Rapikan_Spasi:
    for col in df.columns: df[col] = df[col].str.strip()
for col in df.columns:
    df[col] = df[col].replace(Penanda_Nilai_Kosong, pd.NA)
for col in Kolom_Huruf_Kecil:
    if col not in df: raise KeyError(f"Kolom tidak ditemukan: {col}")
    df[col] = df[col].str.lower()
for col in Kolom_Tanggal:
    if col not in df: raise KeyError(f"Kolom tanggal tidak ditemukan: {col}")
    parsed = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
    print(f"{col}: {parsed.isna().sum():,} nilai kosong/tidak terbaca sebagai tanggal")
    df[col] = parsed.dt.strftime("%Y-%m-%d").astype("string")
for col in Kolom_Angka:
    if col not in df: raise KeyError(f"Kolom angka tidak ditemukan: {col}")
    # Ganti koma desimal menjadi titik sebelum konversi jika format angka Anda memakai koma.
    parsed = pd.to_numeric(df[col].str.replace(",", ".", regex=False), errors="coerce")
    print(f"{col}: {parsed.isna().sum():,} nilai kosong/tidak terbaca sebagai angka")
    df[col] = parsed
before = len(df)
if Hapus_Baris_Kosong: df = df.dropna(how="all")
print(f"Baris kosong dihapus: {before-len(df):,}; sisa: {len(df):,}")
display(df.head())


In [ ]:
output = Output_CSV if Output_CSV.is_absolute() else BASE_DIR / Output_CSV
output.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output, index=False, encoding="utf-8-sig", sep=Separator_CSV)
print(f"Tersimpan: {output.resolve()}")